# HotpotQA — Inference, Evaluation & Contrastive Training

This notebook runs the full HalluLens pipeline on HotpotQA in **closed-book mode**
(no supporting passages provided — the model relies on parametric memory).

**Dataset:** [HotpotQA](https://huggingface.co/datasets/hotpot_qa) — multi-hop QA requiring
reasoning across two Wikipedia articles per question.
- `validation` split: 7,405 questions (default — good for eval and training the contrastive model)
- `train` split: 90,447 questions (available for large-scale runs)
- Two question types: **bridge** (chain two entities) and **comparison** (compare attributes)

**Why multi-hop matters for hallucination detection:**  
Answering correctly requires chaining two facts. Hallucinations here reflect a different
failure mode (incomplete multi-step reasoning) compared to TriviaQA / NQ (single-fact recall).
Generalisation across these two regimes is a strong paper result.

**Steps:**
1. Inference — generate model responses with activation logging
2. Evaluation — substring match against gold answer; write `eval_results.json`
3. Inspection — per-sample view, breakdown by question type and difficulty
4. Training — contrastive model on HotpotQA activations
5. OOD evaluation — score the trained model

**Prerequisite:** A running vLLM server (`COMPUTE_CONTEXT=REMOTE_GPU` in `.env`).

## 1 — Configuration

In [ ]:
import os, sys, json
from pathlib import Path

# ── Repo root on path ──────────────────────────────────────────────────────
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

# ── Model ──────────────────────────────────────────────────────────────────
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model_name = MODEL.split("/")[-1]

# ── Dataset split and sample count ────────────────────────────────────────
SPLIT = "validation"   # "validation" | "train"
N     = None            # None = entire split

# ── Storage paths ──────────────────────────────────────────────────────────
base_dir          = Path("shared/hotpotqa")
activations_path  = str(base_dir / "activations.zarr")
log_file          = str(base_dir / "server.log")

output_dir        = "output"
task_dir          = Path(output_dir) / "hotpotqa" / model_name
generations_file  = str(task_dir / "generation.jsonl")
eval_results_file     = str(task_dir / "eval_results.json")
eval_results_llm_file = str(task_dir / "eval_results_llm.json")

# ── Activation logging ─────────────────────────────────────────────────────
LOGGER_TYPE = "zarr"   # zarr (preferred) | lmdb | json

# ── Inference settings ─────────────────────────────────────────────────────
MAX_TOKENS  = 128
TEMPERATURE = 0.0

# ── LLM evaluator ──────────────────────────────────────────────────────────
# Used for the LLM-judge eval (eval_llm step).
# Default: neuralmagic-ent/Llama-3.3-70B-Instruct-quantized.w8a8
LLM_EVALUATOR = None   # None → use class default

base_dir.mkdir(parents=True, exist_ok=True)
task_dir.mkdir(parents=True, exist_ok=True)

print(f"Model             : {MODEL}")
print(f"Split             : {SPLIT}  (N={N or 'all'})")
print(f"Generations file  : {generations_file}")
print(f"Eval (substring)  : {eval_results_file}")
print(f"Eval (LLM judge)  : {eval_results_llm_file}")
print(f"Activations       : {activations_path}")
print(f"Logger type       : {LOGGER_TYPE}")

## 2 — Inference

Generates model responses for HotpotQA questions in closed-book mode and logs activations.
Resume-safe: questions already in `generation.jsonl` are skipped.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="inference",
    task="hotpotqa",
    model=MODEL,
    split=SPLIT,
    N=N,
    logger_type=LOGGER_TYPE,
    activations_path=activations_path,
    log_file=log_file,
    output_dir=output_dir,
    generations_file_path=generations_file,
    max_inference_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    resume=True,
)

## 3 — Evaluation

Two evaluation methods are run and compared:

- **Substring match** (`eval`) — fast, deterministic. Correct if the gold answer appears verbatim in the response. Saved to `eval_results.json`.
- **LLM judge** (`eval_llm`) — uses `neuralmagic-ent/Llama-3.3-70B-Instruct-quantized.w8a8` to assess semantic correctness. Handles paraphrased answers and yes/no polarity for comparison questions. Saved to `eval_results_llm.json`, which also includes substring results for direct comparison.

The LLM eval is resume-safe (incremental saving to `llm_halu_eval_raw.jsonl`).

In [ ]:
from scripts.run_with_server import run_experiment

# ── 3a: Substring match eval ───────────────────────────────────────────────
run_experiment(
    step="eval",
    task="hotpotqa",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_file,
)

# ── 3b: LLM-judge eval ────────────────────────────────────────────────────
run_experiment(
    step="eval_llm",
    task="hotpotqa",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_llm_file,
    llm_evaluator=LLM_EVALUATOR,
    resume=True,
)

## 4 — Results Inspection & Evaluator Comparison

In [ ]:
import pandas as pd

with open(eval_results_file) as f:
    substr_res = json.load(f)

with open(eval_results_llm_file) as f:
    llm_res = json.load(f)

n = substr_res["total_count"]
llm_correct_count = llm_res["accurate_count"]
substr_correct_count = substr_res["accurate_count"]
agree_count = int(llm_res["evaluator_agreement_rate"] * n)

print(f"Model          : {model_name}")
print(f"Split          : {SPLIT}  (N={n})")
print()
print(f"{'Metric':<30} {'Substring':>12} {'LLM Judge':>12}")
print("-" * 55)
print(f"{'Correct count':<30} {substr_correct_count:>12} {llm_correct_count:>12}")
print(f"{'Correct rate':<30} {substr_res['correct_rate']:>12.1%} {llm_res['correct_rate']:>12.1%}")
print(f"{'Hallucination rate':<30} {substr_res['halu_Rate']:>12.1%} {llm_res['halu_Rate']:>12.1%}")
print("-" * 55)
print(f"{'Evaluator agreement':<30} {llm_res['evaluator_agreement_rate']:>12.1%}")
print()
print(f"LLM evaluator  : {llm_res['evaluator_hallucination']}")

In [ ]:
# Build a unified per-sample dataframe combining both evaluators
raw_df = pd.read_json(task_dir / "raw_eval_res.jsonl", lines=True)

# Merge LLM judge verdicts
raw_df["halu_llm"]    = llm_res["halu_test_res"]
raw_df["halu_substr"] = llm_res["substring_halu_test_res"]   # re-computed alongside LLM eval

# Agreement column: True where both evaluators agree
raw_df["agree"] = raw_df["halu_llm"] == raw_df["halu_substr"]

print(f"Loaded {len(raw_df)} samples")
raw_df.head()

In [ ]:
# ── Hallucination rate by question type (bridge vs comparison) ─────────────
if "type" in raw_df.columns:
    type_stats = (
        raw_df.groupby("type")
        .agg(
            count=("halu_llm", "count"),
            halu_rate_substr=("halu_substr", "mean"),
            halu_rate_llm=("halu_llm", "mean"),
            agreement=("agree", "mean"),
        )
        .reset_index()
    )
    for col in ["halu_rate_substr", "halu_rate_llm", "agreement"]:
        type_stats[col] = type_stats[col].map("{:.1%}".format)
    print("Hallucination rate by question type:")
    display(type_stats)

# ── Hallucination rate by difficulty level ─────────────────────────────────
if "level" in raw_df.columns:
    level_stats = (
        raw_df.groupby("level")
        .agg(
            count=("halu_llm", "count"),
            halu_rate_substr=("halu_substr", "mean"),
            halu_rate_llm=("halu_llm", "mean"),
            agreement=("agree", "mean"),
        )
        .reset_index()
    )
    for col in ["halu_rate_substr", "halu_rate_llm", "agreement"]:
        level_stats[col] = level_stats[col].map("{:.1%}".format)
    print("\nHallucination rate by difficulty level:")
    display(level_stats)

In [ ]:
# ── Evaluator disagreements (where methods diverge) ────────────────────────
# These are the most informative cases: substring says wrong, LLM says correct.
# Indicates substring is under-counting correct answers (paraphrase failures).
disagree_df = raw_df[~raw_df["agree"]].reset_index(drop=True)
false_halu  = raw_df[raw_df["halu_substr"] & ~raw_df["halu_llm"]].reset_index(drop=True)
false_corr  = raw_df[~raw_df["halu_substr"] & raw_df["halu_llm"]].reset_index(drop=True)

print(f"Total disagreements : {len(disagree_df)}  ({len(disagree_df)/max(1,len(raw_df)):.1%})")
print(f"  Substr=halu, LLM=correct (likely false hallucinations) : {len(false_halu)}")
print(f"  Substr=correct, LLM=halu (LLM over-penalising)         : {len(false_corr)}")

print("\n── Top examples: Substring says hallucinated, LLM says correct ──")
for i, row in false_halu.head(6).iterrows():
    print(f"\n[{row.get('type','')} / {row.get('level','')}]")
    print(f"  Q  : {row['question']}")
    print(f"  Gen: {row['generation']}")
    print(f"  Ans: {row['answer']}")

## 4b — Evaluator Summary

Consolidated view of how much the choice of evaluator matters and where the gap comes from.

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
summary_rows = []

for qtype in raw_df["type"].unique() if "type" in raw_df.columns else []:
    sub = raw_df[raw_df["type"] == qtype]
    summary_rows.append({
        "group": f"type={qtype}",
        "n": len(sub),
        "substr_correct%": f"{(~sub['halu_substr']).mean():.1%}",
        "llm_correct%":    f"{(~sub['halu_llm']).mean():.1%}",
        "agreement%":      f"{sub['agree'].mean():.1%}",
        "false_halu_n":    int((sub['halu_substr'] & ~sub['halu_llm']).sum()),
        "false_corr_n":    int((~sub['halu_substr'] & sub['halu_llm']).sum()),
    })

for level in raw_df["level"].unique() if "level" in raw_df.columns else []:
    sub = raw_df[raw_df["level"] == level]
    summary_rows.append({
        "group": f"level={level}",
        "n": len(sub),
        "substr_correct%": f"{(~sub['halu_substr']).mean():.1%}",
        "llm_correct%":    f"{(~sub['halu_llm']).mean():.1%}",
        "agreement%":      f"{sub['agree'].mean():.1%}",
        "false_halu_n":    int((sub['halu_substr'] & ~sub['halu_llm']).sum()),
        "false_corr_n":    int((~sub['halu_substr'] & sub['halu_llm']).sum()),
    })

# Overall row
summary_rows.insert(0, {
    "group": "ALL",
    "n": len(raw_df),
    "substr_correct%": f"{(~raw_df['halu_substr']).mean():.1%}",
    "llm_correct%":    f"{(~raw_df['halu_llm']).mean():.1%}",
    "agreement%":      f"{raw_df['agree'].mean():.1%}",
    "false_halu_n":    int((raw_df['halu_substr'] & ~raw_df['halu_llm']).sum()),
    "false_corr_n":    int((~raw_df['halu_substr'] & raw_df['halu_llm']).sum()),
})

summary_df = pd.DataFrame(summary_rows)
print("Evaluator comparison summary")
print("  false_halu_n : substring=halu, LLM=correct  → substring under-counts correct answers")
print("  false_corr_n : substring=correct, LLM=halu  → LLM over-penalises\n")
display(summary_df)

## 5 — Contrastive Training

Trains a `ProgressiveCompressor` on HotpotQA activations.
Requires `activations_path` to exist (logged during inference).

In [ ]:
import torch
from loguru import logger as _logger
_logger.remove()
_logger.add(sys.stdout, level="INFO")

from activation_logging.activation_parser import ActivationParser
from activation_research.model import ProgressiveCompressor
from activation_research.training import train_contrastive
from activation_research.metric_evaluator import MultiMetricHallucinationEvaluator
from torch.utils.data import DataLoader

_act = Path(activations_path)
if not _act.exists():
    raise FileNotFoundError(
        f"Activations not found: {activations_path}\n"
        "Re-run inference with activation logging enabled."
    )
print(f"Activations found: {_act.resolve()}")

In [ ]:
# ── Training config ────────────────────────────────────────────────────────
relevant_layers = list(range(14, 30))
target_layers   = [22, 26]
num_views       = 4
outlier_class   = 1

input_dim      = 4096
final_dim      = 512
max_epochs     = 50
batch_size     = 512
sub_batch_size = 64
lr             = 1e-5
temperature    = 0.25
num_workers    = 4

checkpoint_dir = "checkpoints/contrastive_hotpotqa"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ── Load datasets ──────────────────────────────────────────────────────────
ap = ActivationParser(
    inference_json=generations_file,
    eval_json=eval_results_file,
    activations_path=activations_path,
    logger_type=LOGGER_TYPE,
    verbose=False,
)

train_dataset = ap.get_dataset(
    "train",
    relevant_layers=relevant_layers,
    num_views=num_views,
    backend=LOGGER_TYPE,
    preload=True,
)
test_dataset = ap.get_dataset(
    "test",
    relevant_layers=relevant_layers,
    num_views=num_views,
    backend=LOGGER_TYPE,
    preload=True,
)

print(f"Train: {len(train_dataset)}  |  Test: {len(test_dataset)}")
print(ap.df["halu"].value_counts(dropna=False).rename(index={0: "non-halu", 1: "halu"}))

In [ ]:
# ── Build and train model ──────────────────────────────────────────────────
model = ProgressiveCompressor(
    input_dim=input_dim,
    final_dim=final_dim,
    input_dropout=0.3,
)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

train_contrastive(
    model,
    train_dataset,
    test_dataset=test_dataset,
    epochs=max_epochs,
    batch_size=batch_size,
    sub_batch_size=sub_batch_size,
    lr=lr,
    temperature=temperature,
    device=device,
    num_workers=num_workers,
    persistent_workers=num_workers > 0,
    use_labels=True,
    ignore_label=outlier_class,
    checkpoint_dir=checkpoint_dir,
    snapshot_every=10,
    snapshot_keep_last=5,
)

## 6 — OOD Evaluation

In [ ]:
train_ds_eval = train_dataset.slice_layers(target_layers)
test_ds_eval  = test_dataset.slice_layers(target_layers)

train_loader = DataLoader(train_ds_eval, batch_size=64, shuffle=False)
eval_loader  = DataLoader(test_ds_eval,  batch_size=64, shuffle=False)

model.eval().to(device)

ood_eval = MultiMetricHallucinationEvaluator(
    activation_parser_df=ap.df,
    train_data_loader=train_loader,
    layers=None,
    batch_size=256,
    sub_batch_size=64,
    device=device,
    num_workers=num_workers,
    persistent_workers=False,
    outlier_class=outlier_class,
    metrics=[
        "cosine",
        "mds",
        {
            "metric": "knn",
            "kwargs": {
                "k": 50,
                "metric": "euclidean",
                "calibrate_k": True,
                "k_candidates": [50, 100, 200, 500],
                "max_train_size": 200000,
                "sample_seed": 0,
            },
            "train_selection": "all",
        },
    ],
)

ood_stats = ood_eval.compute(eval_loader, model)
print("OOD metrics (HotpotQA):")
for k, v in ood_stats.items():
    print(f"  {k}: {v:.4f}")